In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

In [2]:
x = torch.rand((4,6))
linear = nn.Linear(6, 10)
x

tensor([[0.0206, 0.1615, 0.0180, 0.7671, 0.0130, 0.1918],
        [0.2170, 0.6940, 0.8556, 0.9938, 0.4811, 0.4403],
        [0.4136, 0.3315, 0.7799, 0.6385, 0.0648, 0.4618],
        [0.3698, 0.6671, 0.5016, 0.1455, 0.2653, 0.4203]])

In [3]:
# GLOBALS 
token_amount = 50
B,T,C  = 4, 8 ,32 # Batch size, time steps (seq length), channels (embedding vector size per token)
seq_len = 8


In [4]:
x = linear(x)
x

tensor([[-0.1530, -0.3031,  0.0184, -0.3638, -0.0749,  0.2580, -0.3400, -0.3402,
         -0.0170, -0.2832],
        [-0.5784, -0.0764, -0.1718, -0.7931, -0.1835, -0.0528, -0.3177, -0.9756,
          0.3682, -0.2008],
        [-0.5261, -0.1331, -0.2876, -0.4684, -0.1835,  0.0080, -0.2397, -0.7736,
          0.1010, -0.1698],
        [-0.6195, -0.1278, -0.5060, -0.4147, -0.0557, -0.1827, -0.0509, -0.6034,
         -0.0646, -0.0510]], grad_fn=<AddmmBackward0>)

In [5]:
ten = 2 * torch.rand((5,5)) - 1
ten

tensor([[ 0.6160, -0.0740,  0.5704, -0.4821, -0.9261],
        [ 0.9134,  0.2262,  0.4477, -0.8718, -0.7254],
        [-0.5134,  0.0159, -0.1102,  0.1884,  0.3983],
        [ 0.3563, -0.6070,  0.5156,  0.7350,  0.5373],
        [-0.5153,  0.3874, -0.7512,  0.0903, -0.6494]])

In [6]:
approx = 'None' # 'tanh'
gelu = nn.GELU()
y = gelu(ten)
y

tensor([[ 0.4503, -0.0348,  0.4083, -0.1518, -0.1641],
        [ 0.7485,  0.1334,  0.3012, -0.1671, -0.1698],
        [-0.1560,  0.0081, -0.0503,  0.1083,  0.2608],
        [ 0.2278, -0.1651,  0.3593,  0.5651,  0.3785],
        [-0.1562,  0.2521, -0.1700,  0.0484, -0.1676]])

In [7]:
relu = nn.ReLU()
y_rel = relu(ten)
y_rel

tensor([[0.6160, 0.0000, 0.5704, 0.0000, 0.0000],
        [0.9134, 0.2262, 0.4477, 0.0000, 0.0000],
        [0.0000, 0.0159, 0.0000, 0.1884, 0.3983],
        [0.3563, 0.0000, 0.5156, 0.7350, 0.5373],
        [0.0000, 0.3874, 0.0000, 0.0903, 0.0000]])

In [8]:
torch.rand((3,6))

tensor([[0.3515, 0.7847, 0.5212, 0.0576, 0.8174, 0.9064],
        [0.1761, 0.8579, 0.3621, 0.8850, 0.5742, 0.5554],
        [0.7136, 0.1997, 0.1086, 0.1067, 0.5903, 0.2961]])

In [9]:
B,T,C  = 4, 8 ,32
x = torch.rand(B,T,C)

In [10]:
x.shape

torch.Size([4, 8, 32])

In [11]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

In [12]:
k = key(x)
q = query(x)
k.shape, q.shape

(torch.Size([4, 8, 16]), torch.Size([4, 8, 16]))

In [13]:
wei = q @ k.transpose(-1, -2)
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [14]:
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[[ 3.2889e-01,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [ 7.2807e-02,  1.4669e-01,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [ 2.9904e-01,  3.1947e-01,  2.2673e-01,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [-5.2525e-01, -2.6943e-01, -6.0172e-01, -4.5867e-01,        -inf,
                 -inf,        -inf,        -inf],
         [ 1.7007e-02,  2.1925e-01,  1.1833e-01, -2.0670e-01,  3.9315e-01,
                 -inf,        -inf,        -inf],
         [-1.6457e-01, -5.2690e-02,  4.8446e-02, -2.5580e-01,  4.2041e-02,
           2.2945e-02,        -inf,        -inf],
         [-1.0208e-01, -1.6750e-01, -2.0499e-01, -2.5461e-01, -6.7840e-02,
          -5.3937e-02, -2.3334e-01,        -inf],
         [-1.6529e-02,  2.3024e-02,  1.1891e-01, -5.0845e-02,  9.2032e-02,
           2.8497e-01, -6.0551e-02,  3.0407e-01]],

In [15]:
wei = F.softmax(wei, dim=1)
wei

tensor([[[0.1703, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1319, 0.1573, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1653, 0.1870, 0.2120, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0725, 0.1038, 0.0926, 0.1602, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1247, 0.1692, 0.1902, 0.2061, 0.3253, 0.0000, 0.0000, 0.0000],
         [0.1040, 0.1289, 0.1773, 0.1963, 0.2289, 0.3100, 0.0000, 0.0000],
         [0.1107, 0.1149, 0.1376, 0.1965, 0.2051, 0.2871, 0.4569, 0.0000],
         [0.1206, 0.1390, 0.1903, 0.2409, 0.2407, 0.4029, 0.5431, 1.0000]],

        [[0.0708, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1259, 0.1219, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1679, 0.1920, 0.1998, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1288, 0.1487, 0.1810, 0.2119, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1119, 0.1181, 0.1405, 0.1853, 0.2294, 0.0000, 0.0000, 0.0000],
         [0.1274, 0.111

In [16]:
wei.shape

torch.Size([4, 8, 8])

In [17]:
v = value(x)

In [18]:
v.shape

torch.Size([4, 8, 16])

In [19]:
att = wei @ v

In [20]:
att.shape

torch.Size([4, 8, 16])

In [21]:
test = torch.ones(5)

In [22]:
test[0:2] = test[:2] + 3

In [23]:
test.std()

tensor(1.6432)

In [24]:
x.shape

torch.Size([4, 8, 32])

In [25]:
emb = nn.Embedding(token_amount, C)

In [26]:
torch.manual_seed(47)
input_sequences = torch.randint(0, token_amount, (B, T))
input_sequences

tensor([[ 5, 48, 49,  4,  0, 21, 36,  1],
        [23,  2, 41, 25,  3, 49, 17,  8],
        [22, 40, 19, 11, 27, 21, 38, 42],
        [45, 28, 49, 49, 47, 20, 35, 40]])

In [27]:
# Here we're doing the indexing. For each sequence example, fetch the corresponding embedding matrix.
# `input_sequences` here should be B examples of T token sequences in which each T element is a token ID not some rand value
input_embedding = emb(input_sequences)
input_embedding.shape # B T C

torch.Size([4, 8, 32])

In [28]:
pos_embedding = nn.Embedding(seq_len, C) # For each position, we have an embedding vector.
pos = torch.arange(0, T)
positional_embeddings = pos_embedding(pos)
positional_embeddings.shape

torch.Size([8, 32])

In [29]:
positional_embeddings[0]

tensor([-0.5540, -0.1677, -1.7382, -0.9734, -0.2886, -0.9845, -0.1295, -0.8175,
         1.0130,  0.3874, -0.4618,  1.3403,  0.8922,  0.2661,  0.2014,  0.6140,
         0.1047, -0.1160, -0.5885,  0.0255, -1.2083,  0.2856,  0.7352, -0.6235,
         0.9029,  0.9697,  0.1173, -0.0266,  0.4993, -0.7550,  0.7418,  1.5704],
       grad_fn=<SelectBackward0>)

In [30]:
input_embedding[0][0]

tensor([ 1.5289, -0.2436,  1.0148,  0.4871,  1.5419, -1.7361, -0.5197,  0.1294,
        -0.5686,  1.4401,  0.9560, -2.9516,  0.4598,  0.6803, -0.1072, -0.6939,
        -0.0156, -0.9294, -0.6686, -0.3600, -0.0827,  0.3027,  0.7940,  0.1067,
         0.9131,  0.9145,  0.2109,  0.9564,  0.3193,  0.1717, -0.3701,  0.8950],
       grad_fn=<SelectBackward0>)

In [31]:
x = input_embedding +  positional_embeddings
f'{x[0][0][0].item():4f}, {(1.3891 + -1.7554):4f}' # == (1.3891 + -1.7554)

'0.974948, -0.366300'

In [32]:
test_tensor = torch.arange(0, 20).view(5, 4)
test_tensor

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])

In [33]:
test_tensor.split(2, 1)

(tensor([[ 0,  1],
         [ 4,  5],
         [ 8,  9],
         [12, 13],
         [16, 17]]),
 tensor([[ 2,  3],
         [ 6,  7],
         [10, 11],
         [14, 15],
         [18, 19]]))

In [34]:
tril = torch.tril(torch.ones(T,T))
mask = tril.masked_fill(tril == 0, float('-inf'))
mask

tensor([[1., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., 1., -inf, -inf, -inf],
        [1., 1., 1., 1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [35]:
@dataclass
class GPTConfig:
    seq_len: int = 8
    embedding_len = 32
    n_heads = 8
    n_blocks = 0

class SelfAttention(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.n_heads = config.n_heads
        self.embedding_len = config.embedding_len
        # Each atten head processes an equal portion of the embedding so the numbers must be compatible.
        assert config.embedding_len % config.n_heads == 0
        self.qkv_lin = nn.Linear(config.embedding_len, config.embedding_len * 3)
        self.proj_lin = nn.Linear(config.embedding_len, config.embedding_len)

        # Add stencil here? better with register buffer so it's no grad

    def forward(self, x):
        # rough steps: get QKV > matmul q with k > get mask > apply mask > softmax 
        # > matmul softmax result (wei) with V -> project? (or is that done before?) 
        B, T, C = x.size() 
        # XXX: Would C being smaller than embed_len cause compatibility issue with n_heads?
        # NO!!! C disappears after linear matmul. But wouldn't it prevent matmul cus of mismatch? (B,C) @ (n_emb, n_emb) if C != n_emb > crash
        assert C <= self.config.embedding_len, "wtf?" 
        qkv = self.qkv_lin(x) # B T EMBED_LEN*3
        q, k, v = qkv.split(self.config.embedding_len, -1) # split over emb dim
        
        # Pre process for multihead attn. 
        # For each example, for each head, for each timestep we have headsize result
        q = q.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2) # (B, n_heads, T, head_size) 
        k = k.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2)
        v = v.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2)
        # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) = (B, n_heads, T, T)?
        wei = q @ k.transpose(-1, -2)
        with torch.no_grad():
            tril = torch.tril(torch.ones(T,T))
            wei = wei.masked_fill(tril == 0, float('-inf'))
            print(wei)

In [36]:
s_attn = SelfAttention(GPTConfig())

In [37]:
x = torch.rand(4, GPTConfig.seq_len, GPTConfig.embedding_len)
x.shape

torch.Size([4, 8, 32])

In [38]:
out = s_attn(x)

RuntimeError: shape '[4, 8, 8, 0]' is invalid for input of size 1024

In [ ]:
math.exp(1)

2.718281828459045

In [ ]:
test_softmax = torch.arange(0, 6).float().view(2,3)
test_softmax

tensor([[0., 1., 2.],
        [3., 4., 5.]])

In [ ]:
F.softmax(test_softmax, dim=0)

tensor([[0.0474, 0.0474, 0.0474],
        [0.9526, 0.9526, 0.9526]])

In [ ]:
tst_view = torch.rand(4,8,8,4)
tst_view.shape

torch.Size([4, 8, 8, 4])

In [ ]:
tst_view.view(4,8,32)

tensor([[[0.8335, 0.3510, 0.7374,  ..., 0.4986, 0.9729, 0.9660],
         [0.8035, 0.8097, 0.3513,  ..., 0.1751, 0.0339, 0.7793],
         [0.0822, 0.3366, 0.8414,  ..., 0.2126, 0.0264, 0.2305],
         ...,
         [0.3901, 0.8549, 0.7311,  ..., 0.5139, 0.6362, 0.9789],
         [0.7353, 0.4125, 0.5198,  ..., 0.0455, 0.6813, 0.9366],
         [0.3041, 0.1002, 0.4681,  ..., 0.0343, 0.2428, 0.8075]],

        [[0.5695, 0.8297, 0.3437,  ..., 0.2393, 0.3704, 0.6779],
         [0.3767, 0.1497, 0.0098,  ..., 0.7605, 0.2431, 0.5804],
         [0.8856, 0.1932, 0.2340,  ..., 0.4430, 0.2799, 0.4387],
         ...,
         [0.6070, 0.9912, 0.8454,  ..., 0.1552, 0.4035, 0.8872],
         [0.5107, 0.5700, 0.2241,  ..., 0.6545, 0.7925, 0.1229],
         [0.5824, 0.3939, 0.4118,  ..., 0.3477, 0.3773, 0.2346]],

        [[0.5576, 0.6844, 0.5592,  ..., 0.7349, 0.4985, 0.6342],
         [0.5589, 0.2784, 0.9481,  ..., 0.1934, 0.6645, 0.8061],
         [0.1573, 0.4511, 0.9218,  ..., 0.7916, 0.2409, 0.

In [ ]:
a = torch.randint(1,5, (2,2))
b = torch.randint(1,5, (2,))
a, b

(tensor([[1, 3],
         [4, 3]]),
 tensor([2, 1]))

In [ ]:
a = torch.randint(1,11, (3,1,3))
a

tensor([[[ 9,  1,  4]],

        [[10,  4,  6]],

        [[ 5, 10,  5]]])

In [ ]:
# torch.topk(a, 1, dim=0)
vals_tensor, indices_tensor = a.topk(1, dim=-1)
vals_tensor, indices_tensor

(tensor([[[ 9]],
 
         [[10]],
 
         [[10]]]),
 tensor([[[0]],
 
         [[0]],
 
         [[1]]]))

In [ ]:
sampling = torch.rand(4,4,5)
sampling

tensor([[[0.8353, 0.6266, 0.2309, 0.8413, 0.9017],
         [0.0997, 0.0843, 0.7229, 0.4912, 0.5008],
         [0.0086, 0.1117, 0.9843, 0.5537, 0.0097],
         [0.2016, 0.3952, 0.8641, 0.9786, 0.9982]],

        [[0.5678, 0.8303, 0.5466, 0.1510, 0.3714],
         [0.4989, 0.6035, 0.7160, 0.1658, 0.6305],
         [0.9044, 0.8954, 0.3396, 0.7829, 0.6020],
         [0.6608, 0.5096, 0.8899, 0.1432, 0.3919]],

        [[0.7101, 0.5054, 0.0013, 0.2828, 0.0726],
         [0.9530, 0.9228, 0.4556, 0.0929, 0.6731],
         [0.9948, 0.7785, 0.6316, 0.5736, 0.3361],
         [0.3766, 0.2125, 0.2158, 0.7904, 0.1163]],

        [[0.1224, 0.9106, 0.5472, 0.4197, 0.6174],
         [0.5278, 0.7779, 0.2010, 0.1283, 0.7544],
         [0.8307, 0.5393, 0.0736, 0.5909, 0.7626],
         [0.0420, 0.4902, 0.3697, 0.8389, 0.4133]]])

In [ ]:
top_vals, top_idx = sampling.topk(2)
top_vals.shape


torch.Size([4, 4, 2])

In [ ]:
top_vals = top_vals.view(16,2)
top_vals


tensor([[0.9017, 0.8413],
        [0.7229, 0.5008],
        [0.9843, 0.5537],
        [0.9982, 0.9786],
        [0.8303, 0.5678],
        [0.7160, 0.6305],
        [0.9044, 0.8954],
        [0.8899, 0.6608],
        [0.7101, 0.5054],
        [0.9530, 0.9228],
        [0.9948, 0.7785],
        [0.7904, 0.3766],
        [0.9106, 0.6174],
        [0.7779, 0.7544],
        [0.8307, 0.7626],
        [0.8389, 0.4902]])

In [ ]:
sample_idx = top_vals.multinomial(1)
sample_idx

tensor([[0],
        [1],
        [1],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [0],
        [0],
        [1]])

In [ ]:
top_idx.shape, top_idx

(torch.Size([4, 4, 2]),
 tensor([[[4, 3],
          [2, 4],
          [2, 3],
          [4, 3]],
 
         [[1, 0],
          [2, 4],
          [0, 1],
          [2, 0]],
 
         [[0, 1],
          [0, 1],
          [0, 1],
          [3, 0]],
 
         [[1, 4],
          [1, 4],
          [0, 4],
          [3, 1]]]))

In [ ]:
x = torch.tensor([[1,2],[3,4]])
x

tensor([[1, 2],
        [3, 4]])

In [ ]:
x.gather(0, torch.tensor([[1],[0]]))

tensor([[3],
        [1]])

In [ ]:
proba_dist = torch.rand(3,3,3)

proba_dist

tensor([[[0.8229, 0.2031, 0.1697],
         [0.3440, 0.7221, 0.5861],
         [0.2359, 0.1123, 0.0641]],

        [[0.1017, 0.2499, 0.8767],
         [0.3932, 0.5220, 0.7646],
         [0.2997, 0.4182, 0.9184]],

        [[0.4845, 0.6471, 0.8710],
         [0.0971, 0.0483, 0.8541],
         [0.1856, 0.2468, 0.9583]]])

In [ ]:
proba_dist

tensor([[[0.8229, 0.0000, 0.0000],
         [0.3440, 0.7221, 0.5861],
         [0.2359, 0.1123, 0.0641]],

        [[0.1017, 0.2499, 0.8767],
         [0.3932, 0.5220, 0.7646],
         [0.2997, 0.4182, 0.9184]],

        [[0.4845, 0.6471, 0.8710],
         [0.0971, 0.0483, 0.8541],
         [0.1856, 0.2468, 0.9583]]])

In [ ]:
proba_dist.view(9,3).multinomial(1)

tensor([[0],
        [2],
        [0],
        [2],
        [0],
        [1],
        [0],
        [2],
        [2]])

In [ ]:
probas = torch.rand(4,8,8)
probas

tensor([[[0.8170, 0.4310, 0.9232, 0.5910, 0.4819, 0.7750, 0.0494, 0.1389],
         [0.7961, 0.7262, 0.2960, 0.6552, 0.1324, 0.4364, 0.9153, 0.1001],
         [0.9804, 0.4660, 0.8022, 0.8772, 0.9769, 0.5359, 0.5536, 0.2415],
         [0.0773, 0.4625, 0.8495, 0.5343, 0.1302, 0.3831, 0.8131, 0.8265],
         [0.1747, 0.9002, 0.3008, 0.0250, 0.5452, 0.7241, 0.1347, 0.6980],
         [0.9654, 0.2491, 0.8830, 0.3255, 0.6728, 0.8577, 0.8540, 0.8510],
         [0.2278, 0.5066, 0.5974, 0.0041, 0.7831, 0.4098, 0.4182, 0.5655],
         [0.0913, 0.6629, 0.9711, 0.1880, 0.5733, 0.9914, 0.5295, 0.6218]],

        [[0.8816, 0.1975, 0.5204, 0.3232, 0.7370, 0.0100, 0.2000, 0.3444],
         [0.2017, 0.4089, 0.1429, 0.4213, 0.2477, 0.4108, 0.0491, 0.1749],
         [0.3575, 0.0399, 0.1748, 0.1632, 0.0771, 0.9797, 0.9856, 0.9972],
         [0.9726, 0.4207, 0.9927, 0.2384, 0.4596, 0.1124, 0.6790, 0.7810],
         [0.2835, 0.7390, 0.5460, 0.3075, 0.9895, 0.3080, 0.8106, 0.3167],
         [0.4916, 0.564

In [ ]:
top_v, top_i = probas.topk(2, dim = -1)
top_v.shape

torch.Size([4, 8, 2])

In [ ]:
top_i.view(4*8, 2)[:2]

tensor([[2, 0],
        [6, 0]])

In [ ]:
picks = top_v.view(4*8, 2).multinomial(1)
picks.shape
picks = picks.view(4, 8, 1)
picks

tensor([[[0],
         [1],
         [0],
         [0],
         [0],
         [0],
         [1],
         [0]],

        [[0],
         [1],
         [0],
         [1],
         [0],
         [1],
         [0],
         [0]],

        [[1],
         [1],
         [1],
         [0],
         [0],
         [0],
         [1],
         [0]],

        [[0],
         [1],
         [1],
         [1],
         [1],
         [1],
         [1],
         [0]]])

In [ ]:
top_i.shape, top_i[:1]

(torch.Size([4, 8, 2]),
 tensor([[[2, 0],
          [6, 0],
          [0, 4],
          [2, 7],
          [1, 5],
          [0, 2],
          [4, 2],
          [5, 2]]]))

In [ ]:
top_i.gather(-1, picks)

tensor([[[2],
         [0],
         [0],
         [2],
         [1],
         [0],
         [2],
         [5]],

        [[0],
         [5],
         [7],
         [0],
         [4],
         [1],
         [1],
         [5]],

        [[3],
         [6],
         [7],
         [7],
         [7],
         [3],
         [0],
         [2]],

        [[0],
         [4],
         [2],
         [6],
         [4],
         [3],
         [6],
         [1]]])

In [ ]:
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0])
y = w * x 
y

tensor([6.], grad_fn=<MulBackward0>)

In [ ]:
y.requires_grad

True

In [ ]:
vals, idx = torch.topk(y, 1)
vals.requires_grad, idx.requires_grad

(True, False)

In [ ]:
i = torch.multinomial(torch.softmax(vals, dim=0), 1)
i.requires_grad

False

In [ ]:
loss = vals[i].sum()
loss.backward()

In [ ]:
print(w.grad)

None


In [ ]:
# w = torch.tensor([2,3,5], dtype=float)
w = torch.rand(2,2)
x = torch.rand(3,2)

In [ ]:
y = x @ w
y.requires_grad

False

In [ ]:
# Pytorch flips timestep and channel dims.
# BTC for logits (input of the CE) becomes BCT.
logits = torch.rand(4,10,5)
y = torch.randint(10, (4,5))
loss = F.cross_entropy(logits, y)

In [ ]:
torch.tensor(5).shape

torch.Size([])

In [ ]:
import numpy as np
lst = list(range(10))
arr = np.arange(10, dtype=np.uint16)
np2torch = torch.from_numpy(arr)
tens = torch.tensor(lst, dtype=torch.uint16)
tens = tens.to('cuda') if torch.cuda.is_available() else tens
# lst

In [39]:
torch.arange(8).unsqueeze(0).repeat(5,1)

tensor([[0, 1, 2, 3, 4, 5, 6, 7],
        [0, 1, 2, 3, 4, 5, 6, 7],
        [0, 1, 2, 3, 4, 5, 6, 7],
        [0, 1, 2, 3, 4, 5, 6, 7],
        [0, 1, 2, 3, 4, 5, 6, 7]])